In [ ]:
# OPTIONAL: Kiểm tra nhanh runtime device.
# Notebook 00 là ingestion/manifest notebook nên không bắt buộc có GPU.
try:
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Đang sử dụng thiết bị: {device}")
except ImportError:
    print("torch chưa được cài trong runtime này. Bỏ qua kiểm tra device; notebook 00 không cần GPU.")


## Harness smoke readiness markers

GitHub repo setup uses `git clone` on first run and `git pull` on later runs, then installs System 1 with `pip install -e`.

Runtime roots used by local, Colab, and Kaggle execution: `AIC_REPO_PARENT`, `AIC_REPO_ROOT`, `AIC_DATA_ROOT`, `AIC_RUNTIME_ROOT`, `AIC_ARTIFACT_ROOT`, and `AIC_HF_REPO_ID`.

Notebook 00 phase00 handoff uses `sync-phase00-ingestion` after HF raw ingest and batch assignment complete.

# Notebook 00C — Local folder streaming HF raw upload + phase00 upload

Phiên bản C dùng workflow local laptop/workstation: dataset zip đã được download sẵn vào một folder trên máy local. Notebook không mount Google Drive, không chạy `drive-shadow`, và không phụ thuộc DriveFS.

```text
Local downloaded dataset folder
→ kiểm tra folder local có zip files
→ stream-standardize-upload-raw:
   scan zip members để lập pairing plan
   extract/probe các video/json theo batch nhỏ theo `RAW_UPLOAD_BATCH_SIZE` trong local scratch
   upload mỗi batch lên Hugging Face raw repo bằng batched commit
   dùng `--min-free-gb`, `--drive-sync-sleep-seconds`, `--cleanup-every-files`, `--cleanup-every-gb` để giữ local scratch an toàn
   ghi progress/report nhỏ
   cleanup scratch ngay sau từng batch
→ kiểm tra HF raw repo
→ ingest từ HF raw repo để tạo processed artifacts local
→ assign-batches tạo batch_manifest.csv + batch_*.txt
→ copy audit reports vào local release manifests
→ upload phase00 ingestion artifacts lên Hugging Face processed repo bằng `sync-phase00-ingestion`
→ kiểm tra HF processed repo theo `phase00_ingestion` layout
```

Thiết kế repo:

```text
AIC26_raw     = versioned canonical raw store
AIC26_release = processed workspace + final release repo
```

Raw repo dùng version prefix:

```text
AIC26_raw/
└── canonical_raw_vXXX/
    ├── raw_videos/
    ├── metadata/
    └── manifests/
        ├── canonical_file_manifest.jsonl
        ├── canonical_import_report.json
        ├── canonical_video_inventory.parquet
        ├── missing_metadata.json
        └── unmatched_metadata.json
```

Processed repo dùng phase layout theo release prefix:

```text
AIC26_release/
└── canonical_release_vXXX/
    ├── phase00_ingestion/
    │   ├── tables/videos.parquet
    │   ├── raw_mapping/media_store_manifest.parquet
    │   ├── manifests/batch_manifest.csv
    │   ├── manifests/batch_*.txt
    │   └── reports/
    │       ├── dataset_report.json
    │       ├── ingestion_errors.jsonl
    │       ├── missing_metadata.json
    │       ├── unmatched_metadata.json
    │       ├── canonical_import_report.json
    │       └── stream_standardize_upload_progress.jsonl
    ├── phase01_structure/
    ├── phase02_features/
    ├── phase03_merged/
    ├── releases/
    ├── checkpoints/
    └── logs/
```

`phase00_ingestion` là output của Notebook 00, chưa phải final runtime release. Final app-ready release cho System 2 nằm trong `AIC26_release/canonical_release_vXXX/releases/competition_dataset_vXXX/`.

Legacy flat layout `canonical_release_vXXX/{manifests,tables,raw_mapping}` đã deprecated; output mới phải dùng `canonical_release_vXXX/phase00_ingestion/{manifests,tables,raw_mapping,reports}`.

Điểm quan trọng:

- `archive_source_dir` là folder local chứa các file `.zip` đã download từ ban tổ chức.
- `scratch_dir` phải nằm trên ổ local còn đủ dung lượng và không được trỏ vào chính folder dataset source.
- Không tạo full `standardize/raw_videos` và `standardize/metadata`; workflow C stream zip trực tiếp lên HF raw rồi cleanup scratch.
- `missing_metadata.json` và `unmatched_metadata.json` là raw-level audit manifests ở `AIC26_raw/canonical_raw_vXXX/manifests/`. `AIC26_release/canonical_release_vXXX/phase00_ingestion/reports/` giữ snapshot/copy cho từng run.
- HF ingest dùng `--canonical-hf-repo-id` và `--canonical-hf-prefix`.
- Notebook chỉ orchestration/verification. Logic storage contract chính nằm trong package CLI.


In [ ]:
import os
import sys
from pathlib import Path
from dataclasses import dataclass

@dataclass
class WorkflowConfig:
    # 1. Hugging Face repos
    # Raw repo: versioned canonical raw store. Use env vars to avoid overwriting prior runs.
    hf_canonical_repo: str = os.environ.get("AIC_HF_RAW_REPO_ID", "1thesudden/AIC26_raw")
    raw_import_id: str = os.environ.get("AIC_RAW_IMPORT_ID", "canonical_raw_v008")

    # Processed repo: release/control plane.
    hf_release_repo: str = os.environ.get("AIC_HF_REPO_ID", "1thesudden/AIC26_release")
    release_id: str = os.environ.get("AIC_RELEASE_ID", "canonical_release_v008")

    # 2. Local dataset paths
    # Set AIC_LOCAL_DATASET_DIR to the folder containing organizer zip files downloaded from the contest source.
    workspace: Path = Path(os.environ.get("AIC_WORKSPACE", Path.cwd())).expanduser().resolve()
    archive_source_dir: str = os.environ.get(
        "AIC_LOCAL_DATASET_DIR",
        str((Path(os.environ.get("AIC_WORKSPACE", Path.cwd())) / "data" / "raw_dataset").expanduser()),
    )
    archive_target_dir: str = os.environ.get("AIC_STREAM_REPORT_DIR", str(workspace / "stream_reports"))
    scratch_dir: str = os.environ.get("AIC_LOCAL_SCRATCH_DIR", str(workspace / "aic_scratch"))
    stream_progress_path: str = os.environ.get(
        "AIC_STREAM_PROGRESS_PATH",
        str(Path(archive_target_dir) / "stream_standardize_upload_progress.jsonl"),
    )
    output_dir: str = os.environ.get("AIC_OUTPUT_DIR", str(workspace / "output"))
    canonical_staging_root: str = os.environ.get("AIC_CANONICAL_STAGING_ROOT", str(workspace / "canonical_staging"))

    # 3. Repo code
    github_repo_url: str = os.environ.get("AIC_GITHUB_REPO_URL", "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git")
    github_branch: str = os.environ.get("AIC_GITHUB_BRANCH", "system1-refactor")
    repo_dir_name: str = os.environ.get("AIC_REPO_DIR_NAME", "Multimodal-Agentic-Retrieval-Engine")
    use_existing_repo: bool = os.environ.get("AIC_USE_EXISTING_REPO", "1") != "0"

    # 4. Execution options
    execution_mode: str = os.environ.get("AIC_EXECUTION_MODE", "bronze_fast")
    num_batches: int = int(os.environ.get("AIC_NUM_BATCHES", "10"))
    min_free_gb: float = float(os.environ.get("AIC_MIN_FREE_GB", "15"))
    drive_sync_sleep_seconds: int = int(os.environ.get("AIC_DRIVE_SYNC_SLEEP_SECONDS", "30"))
    cleanup_every_files: int = int(os.environ.get("AIC_CLEANUP_EVERY_FILES", "100"))
    cleanup_every_gb: float = float(os.environ.get("AIC_CLEANUP_EVERY_GB", "50"))

    run_upload_standardized_raw: bool = os.environ.get("AIC_RUN_UPLOAD_STANDARDIZED_RAW", "1") != "0"
    run_drive_shadow: bool = False
    run_standardize_archives: bool = False
    run_ingest: bool = os.environ.get("AIC_RUN_INGEST", "1") != "0"
    run_assign_batches: bool = os.environ.get("AIC_RUN_ASSIGN_BATCHES", "1") != "0"
    run_upload_processed: bool = os.environ.get("AIC_RUN_UPLOAD_PROCESSED", "1") != "0"

    def __post_init__(self):
        self.env = "local"
        self.workspace.mkdir(parents=True, exist_ok=True)
        os.environ["AIC_RELEASE_ID"] = self.release_id
        os.environ["AIC_HF_REPO_ID"] = self.hf_release_repo

config = WorkflowConfig()

print("Môi trường:", config.env)
print("Workspace:", config.workspace)
print("Release ID:", config.release_id)
print("HF processed repo:", config.hf_release_repo)
print("HF raw repo:", config.hf_canonical_repo)
print("Raw import ID:", config.raw_import_id)
print("Run upload standardized raw:", config.run_upload_standardized_raw)
print("Run drive shadow:", config.run_drive_shadow)
print("Archive source dir local:", config.archive_source_dir)
print("Stream report dir:", config.archive_target_dir)
print("Scratch dir:", config.scratch_dir)
print("Output dir:", config.output_dir)
print("Stream progress path:", config.stream_progress_path)


In [ ]:
# BƯỚC 1: Configure local environment và HF token.
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Notebook 00C chạy local, không mount Google Drive và không gọi Drive API.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ.setdefault("AIC_VERBOSE", "0")
os.environ.setdefault("AIC_HF_PROGRESS", "0")

hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        import getpass
        hf_token = getpass.getpass("HF token (input hidden, leave blank to skip for now): ").strip()
    except Exception:
        hf_token = ""

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["AIC_HF_TOKEN"] = hf_token
    print("HF token configured from environment/input.")
else:
    print("Cảnh báo: chưa thấy HF_TOKEN/AIC_HF_TOKEN. Các bước upload/check HF sẽ fail cho tới khi token được set.")

print("Local workflow: skip Google Drive mount/auth.")


In [ ]:
# BƯỚC 2: Resolve repo code local hoặc clone/sync repo.
from pathlib import Path
import os
import shutil
import subprocess

def looks_like_repo_root(path: Path) -> bool:
    return (path / "system1" / "src" / "system1" / "__init__.py").exists() or (path / "src" / "system1" / "__init__.py").exists()

def find_existing_repo(start: Path) -> Path | None:
    start = start.expanduser().resolve()
    for candidate in [start, *start.parents]:
        if looks_like_repo_root(candidate):
            return candidate
    return None

def run_git(cmd, *, cwd=None, check=True):
    safe_cwd = Path(cwd).expanduser().resolve() if cwd else config.workspace
    safe_cwd.mkdir(parents=True, exist_ok=True)

    print("CWD:", safe_cwd)
    print("RUN:", " ".join(map(str, cmd)))

    result = subprocess.run(
        list(map(str, cmd)),
        cwd=str(safe_cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if result.stdout:
        print(result.stdout)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit_code={result.returncode}\n"
            f"CMD: {' '.join(map(str, cmd))}\n\n"
            f"OUTPUT:\n{result.stdout}"
        )

    return result

existing_repo = find_existing_repo(Path.cwd()) if config.use_existing_repo else None
repo_dir = existing_repo or (config.workspace / config.repo_dir_name).expanduser().resolve()

print("Current cwd:", Path.cwd())
print("workspace:", config.workspace)
print("repo_dir:", repo_dir)
print("use_existing_repo:", config.use_existing_repo)
print("repo_dir exists:", repo_dir.exists())
print("repo_dir .git exists:", (repo_dir / ".git").exists())
print("github_repo_url:", config.github_repo_url)
print("github_branch:", config.github_branch)

if existing_repo is not None:
    print("Dùng repo hiện tại:", existing_repo)
elif (repo_dir / ".git").exists():
    print("Repo đã tồn tại, cập nhật repo:", repo_dir)
    run_git(["git", "fetch", "--all", "--prune"], cwd=repo_dir)
    branch_check = run_git(["git", "rev-parse", "--verify", f"origin/{config.github_branch}"], cwd=repo_dir, check=False)
    if branch_check.returncode != 0:
        raise RuntimeError(f"Không tìm thấy remote branch origin/{config.github_branch}.")
    run_git(["git", "checkout", "-B", config.github_branch, f"origin/{config.github_branch}"], cwd=repo_dir)
    run_git(["git", "reset", "--hard", f"origin/{config.github_branch}"], cwd=repo_dir)
else:
    if repo_dir.exists():
        raise RuntimeError(
            f"repo_dir tồn tại nhưng không phải git repo hoặc không nhận diện được source layout: {repo_dir}. "
            "Hãy set AIC_WORKSPACE/AIC_REPO_DIR_NAME khác hoặc xóa folder này thủ công."
        )
    print("Clone repo:", config.github_repo_url)
    run_git(["git", "clone", config.github_repo_url, str(repo_dir)], cwd=config.workspace)
    run_git(["git", "fetch", "--all", "--prune"], cwd=repo_dir)
    branch_check = run_git(["git", "rev-parse", "--verify", f"origin/{config.github_branch}"], cwd=repo_dir, check=False)
    if branch_check.returncode != 0:
        raise RuntimeError(f"Clone được repo nhưng không tìm thấy origin/{config.github_branch}.")
    run_git(["git", "checkout", "-B", config.github_branch, f"origin/{config.github_branch}"], cwd=repo_dir)

print("Git commit hiện tại:")
run_git(["git", "log", "-1", "--oneline"], cwd=repo_dir, check=False)

print("Git status:")
run_git(["git", "status", "--short"], cwd=repo_dir, check=False)


In [ ]:
# 2B. Kiểm tra project_root và setup import path cho notebook kernel.
import sys
import subprocess
from pathlib import Path

print("\nCheck package source layout:")

project_root_candidates = [
    repo_dir / "system1",
    repo_dir,
]

project_root = None
for candidate in project_root_candidates:
    print("- candidate:", candidate)
    print("  has src/system1:", (candidate / "src" / "system1").exists())
    print("  has src/system1/__init__.py:", (candidate / "src" / "system1" / "__init__.py").exists())
    print("  has runtime/environment.py:", (candidate / "src" / "system1" / "runtime" / "environment.py").exists())

    if (candidate / "src" / "system1" / "__init__.py").exists():
        project_root = candidate.resolve()
        break

if project_root is None:
    raise RuntimeError(
        "Không tìm thấy project_root chứa src/system1/__init__.py.\n"
        "Checked:\n" + "\n".join(str(p) for p in project_root_candidates)
    )

system1_src = project_root / "src"

print("\nInstalling editable package from:", project_root)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(project_root)],
    check=True,
)

# Đưa src path thật lên đầu sys.path để notebook import đúng package.
system1_src_str = str(system1_src)
sys.path = [system1_src_str] + [
    p for p in sys.path
    if str(Path(p).expanduser().resolve()) != system1_src_str
]

# Xóa cache nếu trước đó import nhầm system1 thành namespace package.
for module_name in list(sys.modules):
    if module_name == "system1" or module_name.startswith("system1."):
        del sys.modules[module_name]

import system1

REPO_ROOT = repo_dir
SYSTEM1_ROOT = project_root

print("\nResolved project paths:")
print("- REPO_ROOT:", REPO_ROOT)
print("- SYSTEM1_ROOT:", SYSTEM1_ROOT)
print("- system1_src:", system1_src)
print("- system1 file:", getattr(system1, "__file__", None))
print("- system1 path:", list(getattr(system1, "__path__", [])))

environment_module_path = system1_src / "system1" / "runtime" / "environment.py"
print("- environment.py exists:", environment_module_path.exists())

if getattr(system1, "__file__", None) is None:
    raise RuntimeError(
        "system1 vẫn đang bị import thành namespace package. "
        "Hãy restart runtime rồi chạy lại từ BƯỚC 1."
    )

In [ ]:
# BƯỚC 3: Định nghĩa helper run_cli để gọi system1 CLI trong notebook.
def run_cli(args, *, check=True, stream=True, tail_lines=300):
    import os
    import subprocess
    import sys
    from pathlib import Path

    def find_repo_root():
        candidates = [
            globals().get("SYSTEM1_ROOT"),
            globals().get("REPO_ROOT"),
            Path.cwd(),
            Path(config.workspace) / config.repo_dir_name,
            Path(config.workspace),
        ]

        for candidate in candidates:
            if candidate is None:
                continue
            path = Path(candidate).expanduser().resolve()

            # Nếu đang ở project package system1/.
            if path.name == "system1" and (path / "src" / "system1").exists():
                return path

            # Nếu đang ở repo root có folder system1/src/system1.
            if (path / "system1" / "src" / "system1").exists():
                return path

        raise RuntimeError(
            "Không tìm thấy repo root. Hãy chạy cell clone/setup repo trước, "
            "hoặc kiểm tra AIC_WORKSPACE/AIC_REPO_DIR_NAME có trỏ đúng repo local không."
        )

    cli_cwd = find_repo_root()
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    # Giữ tương thích với bản cũ:
    # - nếu cli_cwd là repo root cha: dùng system1/src
    # - nếu cli_cwd là package root system1/: dùng src
    if (cli_cwd / "src" / "system1").exists():
        system1_src = cli_cwd / "src"
    else:
        system1_src = cli_cwd / "system1" / "src"

    env["PYTHONPATH"] = str(system1_src) + os.pathsep + env.get("PYTHONPATH", "")

    # Chặn progress bar nội bộ của huggingface_hub trong subprocess.
    # Nếu cần debug HF progress bar thật, set AIC_HF_PROGRESS=1 trước khi gọi run_cli.
    if env.get("AIC_HF_PROGRESS", "0") != "1":
        env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        env["HF_HUB_VERBOSITY"] = "error"

    # Mặc định không in per-file/per-cleanup nếu package đã hỗ trợ AIC_VERBOSE.
    env.setdefault("AIC_VERBOSE", "0")

    cmd = [sys.executable, "-m", "system1.cli", *args]

    print("\n" + "=" * 100)
    print("CWD:", cli_cwd)
    print("PYTHONPATH prefix:", system1_src)
    print("RUN:", " ".join(cmd))
    print("stream:", stream)
    print("=" * 100)

    if not stream:
        completed = subprocess.run(
            cmd,
            cwd=str(cli_cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

        output = completed.stdout or ""
        output_lines = output.splitlines()
        tail = "\n".join(output_lines[-tail_lines:])

        print(f"CLI finished: exit_code={completed.returncode}")
        print(f"Last {min(tail_lines, len(output_lines))} lines:")
        print("-" * 100)
        print(tail)
        print("-" * 100)

        if check and completed.returncode != 0:
            raise RuntimeError(
                f"CLI failed with exit code {completed.returncode}: {' '.join(args)}\n\n"
                f"Last output:\n{tail}"
            )

        return output

    process = subprocess.Popen(
        cmd,
        cwd=str(cli_cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)

    returncode = process.wait()
    output = "".join(lines)

    if check and returncode != 0:
        tail = "\n".join(output.splitlines()[-tail_lines:])
        raise RuntimeError(
            f"CLI failed with exit code {returncode}: {' '.join(args)}\n\n"
            f"Last output:\n{tail}"
        )

    return output

In [ ]:
# BƯỚC 4: Resolve runtime paths và kiểm tra local workflow paths.
from system1.runtime.environment import resolve_runtime_paths
from pathlib import Path

runtime_paths = resolve_runtime_paths(output_root=Path(config.output_dir))
output_base = runtime_paths.output_root
output_base.mkdir(parents=True, exist_ok=True)

archive_source = Path(config.archive_source_dir).expanduser().resolve()
stream_report_dir = Path(config.archive_target_dir).expanduser().resolve()
scratch_dir = Path(config.scratch_dir).expanduser().resolve()
canonical_staging_root = Path(config.canonical_staging_root).expanduser().resolve()

print("Runtime paths:")
print("- environment:", runtime_paths.environment)
print("- workspace_root:", runtime_paths.workspace_root)
print("- output_root:", runtime_paths.output_root)
print("- artifact_root:", runtime_paths.artifact_root)
print("- release_id:", runtime_paths.release_id)

print("Local workflow paths:")
print("- archive_source_dir:", archive_source)
print("- stream_report_dir:", stream_report_dir)
print("- scratch_dir:", scratch_dir)
print("- canonical_staging_root:", canonical_staging_root)

if scratch_dir == archive_source or archive_source in scratch_dir.parents:
    raise RuntimeError("scratch_dir không được nằm bên trong archive_source_dir để tránh xóa nhầm dataset source.")
if canonical_staging_root == archive_source or archive_source in canonical_staging_root.parents:
    raise RuntimeError("canonical_staging_root không được nằm bên trong archive_source_dir.")


In [ ]:
# BƯỚC 5: Kiểm tra local dataset folder đã download.
from pathlib import Path
import json
import shutil
import subprocess

archive_source = Path(config.archive_source_dir).expanduser().resolve()
local_source_report_path = Path(config.archive_target_dir).expanduser().resolve() / "local_source_report.json"

print("BƯỚC 5: Check local source folder")
print("- archive_source_dir:", archive_source)
print("- exists:", archive_source.exists())

if not archive_source.exists() or not archive_source.is_dir():
    raise RuntimeError(
        f"Không thấy archive_source_dir local: {archive_source}. "
        "Hãy set AIC_LOCAL_DATASET_DIR hoặc sửa config.archive_source_dir tới folder chứa zip dataset."
    )

source_files = sorted([p for p in archive_source.rglob("*") if p.is_file()])
zip_files = [p for p in source_files if p.suffix.lower() == ".zip"]

print("- source file count:", len(source_files))
print("- zip file count:", len(zip_files))
for p in zip_files[:50]:
    print(" zip:", p)

if not zip_files:
    raise RuntimeError("archive_source_dir không có zip file để stream.")

stream_report_dir = Path(config.archive_target_dir).expanduser().resolve()
stream_report_dir.mkdir(parents=True, exist_ok=True)
local_source_report = {
    "status": "pass",
    "source_dir": str(archive_source),
    "source_file_count": len(source_files),
    "zip_file_count": len(zip_files),
    "zip_files": [p.relative_to(archive_source).as_posix() for p in zip_files],
}
local_source_report_path.write_text(json.dumps(local_source_report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("- local_source_report:", local_source_report_path)

print("Disk overview:")
for path in [archive_source, Path(config.scratch_dir).expanduser(), output_base]:
    subprocess.run(["df", "-h", str(path if path.exists() else path.parent)], check=False)


In [ ]:
# BƯỚC 6: Không dùng Google Drive trong workflow local.
print("Notebook 00C local: skip drive-shadow.")
print("- run_drive_shadow:", config.run_drive_shadow)
if config.run_drive_shadow:
    raise RuntimeError("Notebook 00C phải chạy với config.run_drive_shadow=False.")


In [ ]:
# BƯỚC 7: Không remount Drive; xác nhận lại local zip source.
from pathlib import Path

archive_source = Path(config.archive_source_dir).expanduser().resolve()
zip_files = sorted([p for p in archive_source.rglob("*.zip") if p.is_file()])
print("Local source ready:")
print("- archive_source_dir:", archive_source)
print("- zip_file_count:", len(zip_files))
if not zip_files:
    raise RuntimeError("Không thấy zip files trước khi stream upload.")


In [ ]:
# BƯỚC 8: Stream standardize zip pairs từ local folder và upload HF raw.
# Workflow C không tạo full raw_videos/metadata local; package extract/probe theo batch nhỏ trong scratch, upload HF bằng batched commit, ghi progress rồi cleanup scratch.

from pathlib import Path
import shutil
import subprocess
import sys

archive_source = Path(config.archive_source_dir).expanduser().resolve()
stream_report_dir = Path(config.archive_target_dir).expanduser().resolve()
scratch_dir = Path(config.scratch_dir).expanduser().resolve()
stream_progress_path = Path(config.stream_progress_path).expanduser().resolve()

print("BƯỚC 8: Stream standardize + upload raw to HF")
print("- archive_source:", archive_source)
print("- scratch_dir:", scratch_dir)
print("- stream_report_dir:", stream_report_dir)
print("- stream_progress_path:", stream_progress_path)
print("- hf_canonical_repo:", config.hf_canonical_repo)
print("- raw_import_id:", config.raw_import_id)

if not archive_source.exists():
    raise RuntimeError(f"Không thấy archive_source_dir: {archive_source}")

if scratch_dir == archive_source or archive_source in scratch_dir.parents:
    raise RuntimeError(f"scratch_dir không được nằm trong archive_source_dir: {scratch_dir}")

stream_report_dir.mkdir(parents=True, exist_ok=True)
stream_progress_path.parent.mkdir(parents=True, exist_ok=True)

# Cleanup scratch trước khi chạy; chỉ xóa scratch local, không xóa source/progress.
if scratch_dir.exists():
    print("Cleanup local scratch trước khi stream:", scratch_dir)
    shutil.rmtree(scratch_dir, ignore_errors=True)
scratch_dir.mkdir(parents=True, exist_ok=True)

print("Disk trước stream upload:")
for path in [archive_source, scratch_dir, output_base]:
    subprocess.run(["df", "-h", str(path if path.exists() else path.parent)], check=False)
subprocess.run(f"du -sh {str(scratch_dir)!r} 2>/dev/null || true", shell=True, check=False)

source_files = sorted([p for p in archive_source.rglob("*") if p.is_file()])
zip_files = [p for p in source_files if p.suffix.lower() == ".zip"]
print("- source file count:", len(source_files))
print("- zip file count:", len(zip_files))
for p in zip_files[:30]:
    print(" zip:", p)

if not zip_files:
    raise RuntimeError("archive_source_dir không có zip file để stream.")

help_result = subprocess.run(
    [sys.executable, "-m", "system1.cli", "stream-standardize-upload-raw", "--help"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False,
)
if help_result.returncode != 0:
    raise RuntimeError("Package hiện tại chưa expose stream-standardize-upload-raw. Hãy sync code package trước khi chạy notebook 00C.")
for option in ["--scratch-dir", "--progress-path", "--resume", "--no-overwrite", "--min-free-gb", "--cleanup-every-gb"]:
    if option not in help_result.stdout:
        raise RuntimeError(f"stream-standardize-upload-raw thiếu option bắt buộc: {option}")

if config.run_upload_standardized_raw:
    run_cli([
        "stream-standardize-upload-raw",
        "--source-dir", str(archive_source),
        "--target-hf-repo-id", config.hf_canonical_repo,
        "--raw-import-id", config.raw_import_id,
        "--scratch-dir", str(scratch_dir),
        "--progress-path", str(stream_progress_path),
        "--resume",
        "--no-overwrite",
        "--min-free-gb", str(config.min_free_gb),
        "--drive-sync-sleep-seconds", str(config.drive_sync_sleep_seconds),
        "--cleanup-every-files", str(config.cleanup_every_files),
        "--cleanup-every-gb", str(config.cleanup_every_gb),
    ], stream=True, tail_lines=500)
else:
    print("config.run_upload_standardized_raw=False, skip stream upload raw.")

print("Disk sau stream upload:")
for path in [archive_source, scratch_dir, output_base]:
    subprocess.run(["df", "-h", str(path if path.exists() else path.parent)], check=False)
subprocess.run(f"du -sh {str(scratch_dir)!r} 2>/dev/null || true", shell=True, check=False)

if stream_progress_path.exists():
    lines = stream_progress_path.read_text(encoding="utf-8").splitlines()
    print("stream progress records:", len(lines))
    print("stream progress path:", stream_progress_path)
else:
    print("stream progress path chưa tồn tại.")


In [ ]:
# BƯỚC 9: Kiểm tra progress local sau stream upload.
# Raw repo check đầy đủ nằm ở bước 10. Cell này chỉ xác nhận checkpoint/resume file nhỏ trên local disk.

from pathlib import Path
import json

stream_progress_path = Path(config.stream_progress_path)
scratch_dir = Path(config.scratch_dir)

print("BƯỚC 9: Check stream progress")
print("- stream_progress_path:", stream_progress_path)
print("- scratch_dir:", scratch_dir)

if config.run_upload_standardized_raw:
    if not stream_progress_path.exists():
        raise RuntimeError(f"Không thấy stream progress JSONL: {stream_progress_path}")
    records = [json.loads(line) for line in stream_progress_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    pass_records = [record for record in records if record.get("status") == "pass"]
    failed_records = [record for record in records if record.get("status") == "failed"]
    print("- progress_records:", len(records))
    print("- pass_records:", len(pass_records))
    print("- failed_records:", len(failed_records))
    if failed_records:
        raise RuntimeError(f"Stream upload có failed records, ví dụ: {failed_records[:3]}")
    if not pass_records:
        raise RuntimeError("Stream upload chưa có pair nào status=pass.")
    leftover = sorted(scratch_dir.glob("stream_pair_*")) if scratch_dir.exists() else []
    print("- scratch leftovers:", leftover[:10])
    if leftover:
        raise RuntimeError(f"Scratch còn stream_pair_* sau cleanup: {leftover[:10]}")
else:
    print("config.run_upload_standardized_raw=False, skip stream progress check.")


In [ ]:
# BƯỚC 10: Kiểm tra HF raw repo sau canonical raw upload.
# Gate trước khi chạy HF ingest: raw repo phải có video/metadata + 5 canonical raw manifests.
# Không hardcode số lượng file; kiểm tra theo trạng thái thực tế trên HF raw repo và import report.

from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path
import os
import json
import pandas as pd

repo_id = config.hf_canonical_repo
raw_import_id = config.raw_import_id.strip("/")
prefix = raw_import_id + "/"
token = os.environ.get("HF_TOKEN") or os.environ.get("AIC_HF_TOKEN")
if not token:
    raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF raw repo.")

api = HfApi(token=token)
files = set(api.list_repo_files(repo_id=repo_id, repo_type="dataset", token=token))

raw_videos = sorted(f for f in files if f.startswith(prefix + "raw_videos/"))
metadata = sorted(f for f in files if f.startswith(prefix + "metadata/"))

required_manifests = [
    prefix + "manifests/canonical_file_manifest.jsonl",
    prefix + "manifests/canonical_import_report.json",
    prefix + "manifests/canonical_video_inventory.parquet",
    prefix + "manifests/missing_metadata.json",
    prefix + "manifests/unmatched_metadata.json",
]
missing_manifests = [p for p in required_manifests if p not in files]

print("repo:", repo_id)
print("prefix:", prefix)
print("raw_videos:", len(raw_videos))
print("metadata:", len(metadata))
print("required manifests:")
for p in required_manifests:
    print("-", p, "OK" if p in files else "MISSING")

if len(raw_videos) == 0:
    raise RuntimeError("HF raw repo chưa có raw_videos. Chưa chạy canonical raw upload hoặc upload chưa hoàn tất.")

if len(metadata) == 0:
    raise RuntimeError("HF raw repo chưa có metadata. Chưa chạy canonical raw upload hoặc upload chưa hoàn tất.")

if missing_manifests:
    raise RuntimeError(f"HF raw repo thiếu required canonical manifests: {missing_manifests}")

# Canonical raw upload của package tạo metadata tối thiểu cho video thiếu metadata,
# nên raw_videos và metadata trong HF raw repo phải là video-primary pairs.
if len(raw_videos) != len(metadata):
    raise RuntimeError(
        "HF raw repo có số raw_videos và metadata không khớp. "
        f"raw_videos={len(raw_videos)}, metadata={len(metadata)}. "
        "Cần kiểm tra lại canonical raw upload/canonical manifest."
    )

report_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=prefix + "manifests/canonical_import_report.json",
    token=token,
)
report = json.loads(Path(report_path).read_text(encoding="utf-8"))

print("\ncanonical_import_report:")
print("- status:", report.get("status"))
print("- video_count:", report.get("video_count"))
print("- metadata_count:", report.get("metadata_count"))
print("- uploaded_pair_count:", report.get("uploaded_pair_count"))
print("- error_count:", report.get("error_count"))
print("- inventory_path:", report.get("inventory_path"))
print("- missing_metadata_path:", report.get("missing_metadata_path"))
print("- unmatched_metadata_path:", report.get("unmatched_metadata_path"))

uploaded_pair_count = report.get("uploaded_pair_count")
if uploaded_pair_count is not None:
    uploaded_pair_count = int(uploaded_pair_count)
    if len(raw_videos) != uploaded_pair_count or len(metadata) != uploaded_pair_count:
        raise RuntimeError(
            "HF raw file count không khớp uploaded_pair_count trong canonical_import_report. "
            f"raw_videos={len(raw_videos)}, metadata={len(metadata)}, uploaded_pair_count={uploaded_pair_count}."
        )

inventory_local = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=prefix + "manifests/canonical_video_inventory.parquet",
    token=token,
)
inventory_df = pd.read_parquet(inventory_local)
required_inventory_cols = {
    "video_id",
    "video_filename",
    "metadata_filename",
    "video_size_bytes",
    "metadata_size_bytes",
    "canonical_backend",
    "canonical_repo_id",
    "canonical_repo_type",
    "canonical_revision",
    "canonical_prefix",
    "canonical_video_path",
    "canonical_metadata_path",
    "duration_sec",
    "fps",
    "frame_count",
}
missing_inventory_cols = sorted(required_inventory_cols - set(inventory_df.columns))
print("\ncanonical_video_inventory:")
print("- rows:", len(inventory_df))
print("- missing_inventory_cols:", missing_inventory_cols)
display(inventory_df.head())

if missing_inventory_cols:
    raise RuntimeError(f"canonical_video_inventory.parquet thiếu cột required: {missing_inventory_cols}")

if len(inventory_df) != len(raw_videos):
    raise RuntimeError(
        "canonical_video_inventory row count không khớp raw_videos. "
        f"inventory_rows={len(inventory_df)}, raw_videos={len(raw_videos)}."
    )

for audit_name in ["missing_metadata.json", "unmatched_metadata.json"]:
    audit_local = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=prefix + f"manifests/{audit_name}",
        token=token,
    )
    audit_payload = json.loads(Path(audit_local).read_text(encoding="utf-8"))
    print(f"\n{audit_name}:")
    print("- kind:", audit_payload.get("kind"))
    print("- count:", audit_payload.get("count"))

if int(report.get("error_count", 0) or 0) != 0:
    raise RuntimeError("canonical_import_report còn error_count > 0. Hãy kiểm tra report và rerun canonical raw upload.")

print("\nHF raw repo versioned structure OK. Có thể chạy ingest từ HF.")


In [ ]:
# BƯỚC 11: Ingest từ HF raw repo để tạo processed artifacts local.
# Lưu ý: đây là HF canonical ingest, không phải local ingest từ Drive.
from pathlib import Path
import os
import shutil
import subprocess

release_root = output_base / config.release_id
canonical_staging_root = Path(config.canonical_staging_root).expanduser().resolve()

print("BƯỚC 11: Ingest từ HF raw repo")
print("- hf_canonical_repo:", config.hf_canonical_repo)
print("- raw_import_id:", config.raw_import_id)
print("- canonical_staging_root:", canonical_staging_root)
print("- release_root:", release_root)

# Vì command đang chạy --no-resume, xóa staging cũ để tránh trộn state cũ.
# Nếu package sau này có checkpoint/resume thật cho HF ingest, có thể đổi policy này.
if canonical_staging_root.exists():
    shutil.rmtree(canonical_staging_root, ignore_errors=True)
canonical_staging_root.mkdir(parents=True, exist_ok=True)

print("Disk/cache trước HF ingest:")
subprocess.run(["df", "-h", str(canonical_staging_root.parent if not canonical_staging_root.exists() else canonical_staging_root)], check=False)
subprocess.run(f"du -sh {str(canonical_staging_root)!r} 2>/dev/null || true", shell=True, check=False)
subprocess.run("du -sh /root/.cache/huggingface 2>/dev/null || true", shell=True, check=False)

INGEST_MAX_WORKERS = os.environ.get("AIC_INGEST_MAX_WORKERS", "1")
os.environ["AIC_INGEST_MAX_WORKERS"] = INGEST_MAX_WORKERS
print("- ingest max workers:", INGEST_MAX_WORKERS)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

run_cli([
    "ingest",
    "--mode", config.execution_mode,
    "--output", str(output_base),
    "--canonical-hf-repo-id", config.hf_canonical_repo,
    "--canonical-hf-prefix", config.raw_import_id,
    "--canonical-staging-root", str(canonical_staging_root),
    "--max-workers", INGEST_MAX_WORKERS,
    "--no-resume",
], stream=False, tail_lines=300)

videos_found = sorted(output_base.rglob("videos.parquet"))
media_manifest_found = sorted(output_base.rglob("media_store_manifest.parquet"))

print("videos.parquet found:", videos_found)
print("media_store_manifest.parquet found:", media_manifest_found)

if not videos_found:
    raise RuntimeError("Không tìm thấy videos.parquet sau ingest.")
if not media_manifest_found:
    raise RuntimeError("Không tìm thấy media_store_manifest.parquet sau ingest.")

In [ ]:
# BƯỚC 12: Chia batch từ videos.parquet đã tạo sau ingest.
# Lưu ý: workflow hiện tại tạo videos.parquet từ HF raw ingest.
# Command assign-batches hiện tại tự đọc từ output_base theo release/mode.

print("BƯỚC 12: Assign batches")
print("- output_base:", output_base)
print("- execution_mode:", config.execution_mode)
print("- num_batches:", config.num_batches)

videos_found = sorted(output_base.rglob("videos.parquet"))
media_manifest_found = sorted(output_base.rglob("media_store_manifest.parquet"))

print("videos.parquet found:", videos_found)
print("media_store_manifest.parquet found:", media_manifest_found)

if not videos_found:
    raise RuntimeError("Chưa có videos.parquet. Hãy chạy ingest thành công trước khi assign batches.")

if config.run_assign_batches:
    run_cli([
        "assign-batches",
        "--mode", config.execution_mode,
        "--num-batches", str(config.num_batches),
        "--output", str(output_base),
        "--no-resume",
    ], stream=False, tail_lines=300)
else:
    print("Bỏ qua assign-batches theo config.")

batch_manifest_found = sorted(output_base.rglob("batch_manifest.csv"))
batch_txt_found = sorted(output_base.rglob("batch_*.txt"))

print("batch_manifest.csv found:", batch_manifest_found)
print("batch_*.txt found:", batch_txt_found[:30])

if not batch_manifest_found:
    raise RuntimeError("Không tìm thấy batch_manifest.csv sau assign-batches.")

if not batch_txt_found:
    raise RuntimeError("Không tìm thấy batch_*.txt sau assign-batches.")

if len(batch_txt_found) != config.num_batches:
    raise RuntimeError(
        f"Số batch_*.txt không khớp: expected={config.num_batches}, actual={len(batch_txt_found)}"
    )

print("Assign batches OK.")

In [ ]:
# BƯỚC 13: Gom audit reports vào local release manifests.
# sync-phase00-ingestion sẽ map report files sang phase00_ingestion/reports/.

from pathlib import Path
import os
import shutil

from huggingface_hub import hf_hub_download

release_root = output_base / config.release_id
manifests_root = release_root / "manifests"
manifests_root.mkdir(parents=True, exist_ok=True)

required_local_reports = []
optional_local_reports = [
    Path(config.stream_progress_path).expanduser().resolve(),
    Path(config.archive_target_dir).expanduser().resolve() / "local_source_report.json",
]

missing_required_reports = [p for p in required_local_reports if not p.exists()]
if missing_required_reports:
    raise RuntimeError(
        "Thiếu required local audit reports trước khi upload processed artifacts: "
        + ", ".join(str(p) for p in missing_required_reports)
    )

for report_path in required_local_reports + optional_local_reports:
    if report_path.exists():
        target = manifests_root / report_path.name
        shutil.copy2(report_path, target)
        print("Copied local report:", report_path, "->", target)
    else:
        print("Optional local report not found, skip:", report_path)

# Snapshot raw-level canonical import report từ AIC26_raw sang phase00 reports.
token = os.environ.get("HF_TOKEN") or os.environ.get("AIC_HF_TOKEN")
if not token:
    raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để snapshot raw canonical reports.")

raw_prefix = config.raw_import_id.strip("/")
raw_reports = [
    "canonical_import_report.json",
]
for report_name in raw_reports:
    local_path = hf_hub_download(
        repo_id=config.hf_canonical_repo,
        repo_type="dataset",
        filename=f"{raw_prefix}/manifests/{report_name}",
        token=token,
    )
    target = manifests_root / report_name
    shutil.copy2(local_path, target)
    print("Copied raw HF report:", f"{raw_prefix}/manifests/{report_name}", "->", target)

print("manifests:")
for p in sorted(manifests_root.iterdir()):
    print(p)

print("Release root:", release_root)
print("Release files:")
for p in sorted(release_root.rglob("*"))[:100]:
    print(" ", p)


In [ ]:
# BƯỚC 14: Upload phase00 ingestion artifacts lên Hugging Face processed repo.
# Không upload raw_videos/metadata ở bước này.
# Raw media thật đã nằm trong HF raw repo. Processed repo chỉ nhận phase00 tables/raw_mapping/manifests/reports.

from pathlib import Path
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

release_root = output_base / config.release_id
processed_repo_id = config.hf_release_repo
phase00_remote_root = f"{config.release_id}/phase00_ingestion"

print("BƯỚC 14: Upload phase00 ingestion artifacts")
print("- run_upload_processed:", config.run_upload_processed)
print("- processed_repo_id:", processed_repo_id)
print("- release_root:", release_root)
print("- phase00_remote_root:", phase00_remote_root)

if not config.run_upload_processed:
    print("Bỏ qua upload phase00 ingestion artifacts theo config.")
else:
    if not release_root.exists():
        raise RuntimeError(f"Không thấy release_root: {release_root}")

    required_local_files = [
        release_root / "tables" / "videos.parquet",
        release_root / "raw_mapping" / "media_store_manifest.parquet",
        release_root / "manifests" / "batch_manifest.csv",
        release_root / "manifests" / "dataset_report.json",
        release_root / "manifests" / "ingestion_errors.jsonl",
        release_root / "manifests" / "missing_metadata.json",
        release_root / "manifests" / "unmatched_metadata.json",
    ]


    if config.run_upload_standardized_raw:
        required_local_files.extend([
            release_root / "manifests" / "canonical_import_report.json",
            release_root / "manifests" / "stream_standardize_upload_progress.jsonl",
        ])

    missing_local = [p for p in required_local_files if not p.exists()]
    if missing_local:
        raise RuntimeError(
            "Thiếu local phase00 files trước khi sync lên HF: "
            + ", ".join(str(p) for p in missing_local)
        )

    batch_txt_files = sorted((release_root / "manifests").glob("batch_*.txt"))
    if not batch_txt_files:
        raise RuntimeError(f"Không thấy batch_*.txt trong {release_root / 'manifests'}")

    print("Local phase00 files ready:")
    for p in required_local_files:
        print("-", p)
    print("- batch_*.txt count:", len(batch_txt_files))

    run_cli([
        "sync-phase00-ingestion",
        "--output", str(output_base),
        "--hf-repo-id", processed_repo_id,
    ], stream=False, tail_lines=300)

    print("Uploaded phase00 ingestion artifacts to HF:", processed_repo_id)


In [ ]:
# BƯỚC 15: Kiểm tra cấu trúc HF processed repo theo phase00_ingestion layout.
# Layout mới:
# AIC26_release/<release_id>/phase00_ingestion/{tables,raw_mapping,manifests,reports}
# Legacy flat layout <release_id>/{tables,raw_mapping,manifests} chỉ được coi là deprecated.

from pathlib import Path
import os
import pandas as pd

try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi, hf_hub_download

repo_id = config.hf_release_repo
release_id = config.release_id
phase00_root = f"{release_id}/phase00_ingestion"

hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF repo.")

api = HfApi(token=hf_token)
files = api.list_repo_files(repo_id=repo_id, repo_type="dataset", token=hf_token)

release_files = sorted([f for f in files if f.startswith(f"{release_id}/")])
phase00_files = sorted([f for f in files if f.startswith(f"{phase00_root}/")])

print("HF processed repo:", repo_id)
print("release_id:", release_id)
print("phase00_root:", phase00_root)
print("release_file_count:", len(release_files))
print("phase00_file_count:", len(phase00_files))

for f in phase00_files:
    print(" ", f)

required_exact = [
    f"{phase00_root}/tables/videos.parquet",
    f"{phase00_root}/raw_mapping/media_store_manifest.parquet",
    f"{phase00_root}/reports/dataset_report.json",
    f"{phase00_root}/reports/ingestion_errors.jsonl",
    f"{phase00_root}/reports/missing_metadata.json",
    f"{phase00_root}/reports/unmatched_metadata.json",
    f"{phase00_root}/manifests/batch_manifest.csv",
]


if config.run_upload_standardized_raw:
    required_exact.extend([
        f"{phase00_root}/reports/canonical_import_report.json",
        f"{phase00_root}/reports/stream_standardize_upload_progress.jsonl",
    ])

missing_required = [p for p in required_exact if p not in phase00_files]

batch_txt_files = sorted([
    p for p in phase00_files
    if p.startswith(f"{phase00_root}/manifests/batch_") and p.endswith(".txt")
])

forbidden_patterns = [
    "/raw_videos/",
    "/metadata/",
    "/temp_extract/",
    "member_stage_",
    "member_extract_",
]

forbidden_files = [
    p for p in release_files
    if any(pattern in p for pattern in forbidden_patterns)
    or p.lower().endswith((".mp4", ".mov", ".mkv", ".avi", ".webm", ".wav", ".zip"))
]

legacy_flat_files = sorted([
    p for p in release_files
    if p.startswith(f"{release_id}/tables/")
    or p.startswith(f"{release_id}/raw_mapping/")
    or p.startswith(f"{release_id}/manifests/")
])

print("\nCHECK REQUIRED:")
print("- missing_required:", missing_required)
print("- batch_txt_count:", len(batch_txt_files))
print("- batch_txt_files:", batch_txt_files)

print("\nCHECK FORBIDDEN:")
print("- forbidden_files:", forbidden_files)

print("\nCHECK LEGACY FLAT LAYOUT:")
print("- legacy_flat_files_count:", len(legacy_flat_files))
if legacy_flat_files:
    print("WARNING: Repo còn legacy flat layout đã deprecated. Không fail để giữ backward compatibility, nhưng output mới nên dùng phase00_ingestion.")
    for p in legacy_flat_files[:50]:
        print(" legacy:", p)

if missing_required:
    raise RuntimeError(f"Thiếu required files trên HF processed repo phase00_ingestion: {missing_required}")

if len(batch_txt_files) == 0:
    raise RuntimeError("Không thấy phase00_ingestion/manifests/batch_*.txt trên HF processed repo.")

if forbidden_files:
    raise RuntimeError(f"HF processed repo có file không nên upload: {forbidden_files}")

# Kiểm tra nội dung nếu raw upload đã bật:
# media_store_manifest phải có các cột canonical cần thiết.
# Với raw version/prefix, chấp nhận một trong hai:
# - canonical_import_id: tên cũ/notebook cũ
# - canonical_prefix: tên hiện tại trong package
if config.run_upload_standardized_raw:
    media_manifest_local = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=f"{phase00_root}/raw_mapping/media_store_manifest.parquet",
        token=hf_token,
    )

    media_df = pd.read_parquet(media_manifest_local)

    required_canonical_cols = {
        "canonical_backend",
        "canonical_repo_id",
        "canonical_repo_type",
        "canonical_video_path",
        "canonical_metadata_path",
    }

    missing_canonical_cols = sorted(required_canonical_cols - set(media_df.columns))

    has_import_id = "canonical_import_id" in media_df.columns
    has_prefix = "canonical_prefix" in media_df.columns

    print("\nCHECK CANONICAL COLUMNS:")
    print("- columns:", list(media_df.columns))
    print("- missing_canonical_cols:", missing_canonical_cols)
    print("- has canonical_import_id:", has_import_id)
    print("- has canonical_prefix:", has_prefix)

    if missing_canonical_cols:
        raise RuntimeError(f"media_store_manifest thiếu canonical columns: {missing_canonical_cols}")

    if not (has_import_id or has_prefix):
        raise RuntimeError(
            "media_store_manifest thiếu cả canonical_import_id và canonical_prefix. "
            "Cần có ít nhất một cột để xác định HF raw prefix/version."
        )

    if "canonical_import_id" not in media_df.columns and "canonical_prefix" in media_df.columns:
        media_df["canonical_import_id"] = media_df["canonical_prefix"]

    print("- canonical prefix/import sample:")
    display(media_df[[
        "video_id",
        "canonical_repo_id",
        "canonical_repo_type",
        "canonical_import_id",
        "canonical_video_path",
        "canonical_metadata_path",
    ]].head())

print("\nHF processed phase00_ingestion structure OK.")


In [ ]:
# BƯỚC 16: Preview output của notebook 00.

from pathlib import Path
import pandas as pd

videos_found = sorted(output_base.rglob("videos.parquet"))
batch_manifest_found = sorted(output_base.rglob("batch_manifest.csv"))
media_manifest_found = sorted(output_base.rglob("media_store_manifest.parquet"))
batch_txt_found = sorted(output_base.rglob("batch_*.txt"))

print("Preview notebook 00 outputs")
print("- videos.parquet found:", videos_found)
print("- media_store_manifest.parquet found:", media_manifest_found)
print("- batch_manifest.csv found:", batch_manifest_found)
print("- batch_*.txt count:", len(batch_txt_found))

if not videos_found:
    raise RuntimeError("Không tìm thấy videos.parquet.")
if not media_manifest_found:
    raise RuntimeError("Không tìm thấy media_store_manifest.parquet.")
if not batch_manifest_found:
    raise RuntimeError("Không tìm thấy batch_manifest.csv.")
if not batch_txt_found:
    raise RuntimeError("Không tìm thấy batch_*.txt.")

videos_path = videos_found[0]
media_manifest_path = media_manifest_found[0]
batch_manifest_path = batch_manifest_found[0]

videos_df = pd.read_parquet(videos_path)
media_df = pd.read_parquet(media_manifest_path)
batch_df = pd.read_csv(batch_manifest_path)

print("\nvideos.parquet:", videos_path)
print("video_count:", len(videos_df))
display(videos_df.head())

print("\nmedia_store_manifest.parquet:", media_manifest_path)
print("media_manifest_rows:", len(media_df))
display(media_df.head())

print("\nbatch_manifest.csv:", batch_manifest_path)
print("batch_rows:", len(batch_df))
display(batch_df.head())

print("\nBatch txt files:")
for p in batch_txt_found[:30]:
    print(" ", p)

print("\nNotebook 00 hoàn tất nếu cell này chạy xong.")

In [ ]:
# BƯỚC 17 OPTIONAL: Kiểm tra lại HF raw repo versioned sau toàn bộ notebook.
import os

try:
    from huggingface_hub import HfApi
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi

raw_repo_id = config.hf_canonical_repo
raw_import_id = config.raw_import_id.strip("/")

print("BƯỚC 17 OPTIONAL: Check HF raw repo")
print("- run_upload_standardized_raw:", config.run_upload_standardized_raw)
print("- raw_repo_id:", raw_repo_id)
print("- raw_import_id:", raw_import_id)

if not config.run_upload_standardized_raw:
    print("config.run_upload_standardized_raw=False, skip HF raw repo check.")
else:
    hf_token = os.environ.get("AIC_HF_TOKEN") or os.environ.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("Thiếu HF_TOKEN/AIC_HF_TOKEN để kiểm tra HF raw repo.")

    api = HfApi(token=hf_token)
    files = sorted(api.list_repo_files(repo_id=raw_repo_id, repo_type="dataset", token=hf_token))

    prefix = f"{raw_import_id}/"
    prefix_files = [f for f in files if f.startswith(prefix)]

    raw_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/raw_videos/")]
    metadata_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/metadata/")]
    manifest_files = [f for f in prefix_files if f.startswith(f"{raw_import_id}/manifests/")]

    print("prefix_file_count:", len(prefix_files))
    print("raw_videos count:", len(raw_files))
    print("metadata count:", len(metadata_files))
    print("manifests:")
    for f in manifest_files:
        print(" ", f)

    required = [
        f"{raw_import_id}/manifests/canonical_file_manifest.jsonl",
        f"{raw_import_id}/manifests/canonical_import_report.json",
        f"{raw_import_id}/manifests/canonical_video_inventory.parquet",
        f"{raw_import_id}/manifests/missing_metadata.json",
        f"{raw_import_id}/manifests/unmatched_metadata.json",
    ]

    missing = [p for p in required if p not in files]

    forbidden = [
        f for f in prefix_files
        if "standardize_progress.jsonl" in f
        or "standardize_archives_report.json" in f
        or "drive_shadow_report.json" in f
        or "batch_" in f
        or f.endswith("videos.parquet")
        or f.endswith("media_store_manifest.parquet")
    ]

    print("missing required:", missing)
    print("forbidden files:", forbidden)

    if not raw_files:
        raise RuntimeError("HF raw repo không có raw_videos.")
    if not metadata_files:
        raise RuntimeError("HF raw repo không có metadata.")
    if missing:
        raise RuntimeError(f"HF raw repo thiếu required manifests: {missing}")
    if forbidden:
        raise RuntimeError(f"HF raw repo có file không đúng mục đích: {forbidden}")

    print("HF raw repo versioned structure OK.")
